# Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behaviour with logging , analytics, and debugging.
- Tranforming prompts, tool seletion and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

## Summarization Middle Ware
Automaticall summarize conversation history when approaching token limits, preserving recent messages while compressing older contect. SUmmarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Number of Messages

In [4]:
### Message based summarization 
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [7]:
### Run with thread id

config={"configurable": {"thread_id":"test-1"}}

In [9]:
questions = [
    "what is 2+2",
    "what is 10*5",
    "what is 100/4",
    "what is 15-7",
    "what is 3*3",
    "what is 4*4"
]

for q in questions:
    # Pass input as 1st arg, and config as 2nd arg
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]}, 
        config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='a32bc788-f976-4ec1-af2c-87d9a4ba6a88'), AIMessage(content='<think>\nOkay, so the user is asking what 2 plus 2 is. Hmm, that seems straightforward, but maybe I should double-check. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations in mathematics. When you add two numbers together, you\'re combining their values. So 2 plus 2... Well, counting on fingers, 2 and another 2 makes 4. But wait, is there any context where this might not be the case? Like in different number bases or something? Let me think.\n\nIn base 10, which is the standard decimal system we use every day, 2 + 2 definitely equals 4. If we were in base 3, for example, the digits go 0, 1, 2, and then 10. But adding 2 + 2 in base 3 would be 11, which is 4 in decimal. However, unless specified, we usually assume base 10. The question doesn\'t mention any different base, so it\'s safe

### Token Size

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import  tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa , pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config={"configurable": {"thread_id":"test-1"}}

# Token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4


In [14]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~164 tokens, 6 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a325c5de-f831-4419-b029-22e4d8c16173'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qdn7adgex', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.034344709, 'completion_tokens_details': None, 'prompt_time': 0.010940353, 'prompt_tokens_details': None, 'queue_time': 0.160250194, 'total_time': 0.045285062}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0eb6-7f2a-7882-9ebd-ad496fcbc969-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'qdn7adgex', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

### Fraction

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import  tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config={"configurable": {"thread_id":"test-1"}}

# Token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [20]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction =  tokens / 131072 #llama-3.3-70b-versatile context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])}  msgs")
    print(f"{response['messages']}")

Paris: ~1075 tokens (0.8202%), 13  msgs
[HumanMessage(content="Here is a summary of the conversation to date:\n\n<think>\nOkay, let me start by understanding the user's request. They want me to act as a Context Extraction Assistant, which means my main task is to extract the most relevant and high-quality context from the provided conversation history. The goal is to replace the existing conversation history with this summary to save space and avoid repetition.\n\nFirst, I need to look at the messages provided. The user has given a conversation history where the AI consistently responds with the same three hotels for any city requested. The latest message is from the user asking for a hotel in New York, and the AI used the search_hotels tool with an ID (hybvbzmda) and provided the same three hotels again.\n\nNow, breaking down the instructions, I have to structure the summary into four sections: SESSION INTENT, SUMMARY, ARTIFACTS, and NEXT STEPS. Each section needs to capture the most 

## Human in the Loop Middleware
Pause agent execution for human approval, editing, or rejection of tool calls before they execure. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. databases writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """ Mock function to read an email by its ID. """
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """ Mock function to send adn email. """
    return f"Email sent to {recipient} with subject '{subject}'"

In [23]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [25]:
config={"configurable": {"thread_id":"test-approve"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages":[
            HumanMessage(content="Send an email to john@test.com with subject 'Hello' and body 'How are you?'")
        ]
    },
    config
)

In [26]:
result

{'messages': [HumanMessage(content="Send an email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='60221d0a-6004-484b-9dd2-1eb961976071'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8vc5msnwa', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 312, 'total_tokens': 344, 'completion_time': 0.056712023, 'completion_tokens_details': None, 'prompt_time': 0.016661942, 'prompt_tokens_details': None, 'queue_time': 0.058464473, 'total_time': 0.073373965}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e137c-7166-75d0-a22b-c10522154a69-0', tool_calls=[{'name': 'send_email_tool', 'args': 

In [ ]:
from langgraph.types import Command 

# step 2: Approve
if "__interrupt__" in result:
    print(" || Paused! Approving..")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "approve"}]
            }
        ),
        config
    )

    print(f" Result: {result['messages'][-1].content}")


 || Paused! Approving..
 Result: Note: The `send_email_tool` function is a mock function and does not actually send an email.
